# BE-02 · Ktor fundamentals

[Open in Colab](https://colab.research.google.com/github/BuiltBySwap/engineering-lab/blob/main/docs/topics/BE-02.ipynb)

**Code:** [kaizen/server-ktor](https://github.com/BuiltBySwap/kaizen/tree/main/server-ktor)

**Status card** (edit it; the Progress and Tags pages are built from it):

```yaml
id: BE-02
title: "Ktor fundamentals"
track: backend
code: "kaizen/server-ktor"
date: 2026-10-08
status: partial  # not-started | in-progress | partial | done
score: "3/3"
insight: "A Ktor server is a set of setup functions listed in application.yaml, and a plugin only works when it is installed, not just added to Gradle."
tags: [BE-02, backend, ktor, plugins, routing]
pending:
  - "Write my own English sentences (pragmatic, iterate)"
  - "CTO lens in my own words (Decision / Because / Review when)"
  - "MQ421 in my own words (SSE streaming)"
```

## Goal
Build a small Ktor server with two JSON endpoints and understand each piece.

## What I learned (plain words)
Three places decide everything:

| Place | Decides | Android twin |
|---|---|---|
| `build.gradle.kts` | Which libraries exist | app build file |
| `application.yaml` | The port and which setup functions run | `AndroidManifest.xml` |
| Kotlin files | What the server does | Activities and ViewModels |

- A **module** is a setup function (`fun Application.configureX()`), loaded in the order listed in `application.yaml`.
- A **plugin** (`install(...)`) adds a feature to every request: ContentNegotiation turns objects into JSON, CallLogging logs each request.
- Request path: Netty → CallLogging → Routing → handler → ContentNegotiation → response.
- JSON keys are a contract with every installed app: add fields, do not rename them.

## Examples
```text
$ curl -i http://localhost:8080/health
HTTP/1.1 200 OK
Content-Type: application/json
{"status":"ok"}
```
```text
INFO Application - 200 OK: GET - /health in 19ms
```

## Ktor vs the popular frameworks

The same endpoint in four frameworks. The concept is identical: a method, a path, and code that returns data.

```kotlin
// Ktor
get("/health") { call.respond(Health("ok")) }
```
```kotlin
// Spring Boot (Kotlin)
@RestController
class HealthController {
    @GetMapping("/health")
    fun health() = Health("ok")
}
```
```js
// Express (Node)
app.get('/health', (req, res) => res.json({ status: 'ok' }));
```
```python
# FastAPI (Python)
@app.get("/health")
def health(): return {"status": "ok"}
```

| | Language | Style | Handles many requests with | Best at | Trade-off |
|---|---|---|---|---|---|
| **Ktor** | Kotlin | Small core + opt-in plugins | Coroutines | Light services, Kotlin teams | Smaller ecosystem and hiring pool |
| **Spring Boot** | Java/Kotlin | Batteries included, annotations, DI | Threads (virtual threads supported) | Enterprise, huge ecosystem | Heavier, more "magic" |
| **Express / NestJS** | JS/TS | Minimal (Express), structured (NestJS) | Single-threaded event loop | Fast APIs, npm ecosystem | CPU-heavy work blocks the loop |
| **FastAPI / Django** | Python | Type hints (FastAPI); batteries included (Django) | async/await (FastAPI); mostly sync (Django) | AI integration, fast to write | Python speed |
| **ASP.NET Core** | C# | Middleware and controllers | async/await | High-performance enterprise | Microsoft ecosystem |
| **Go** | Go | Minimal | Goroutines | Simple, fast deployment | Fewer language features |

**The three real differences:** (1) transparent vs "magic": Ktor shows every piece; (2) code vs annotations: Ktor builds routes with a Kotlin DSL; (3) coroutines vs threads or an event loop.

**Popularity** (Stack Overflow Developer Survey 2025, as reported by secondary sources, so treat as rough): Express about 19.9%, ASP.NET Core about 19.7%, Spring Boot about 14.7%, FastAPI about 14.8% (up from 9.9%). Ktor is a niche choice; on JVM backends, Spring Boot is far more common in jobs.

## Does learning Ktor teach every backend?
| Transfers to every backend | Does not transfer |
|---|---|
| HTTP, status codes, headers | Spring's DI container and annotations |
| Routing and the plugin/middleware pipeline | ORM "magic" (JPA, Django) |
| JSON, config, logging, errors | Each language's concurrency model |
| Databases, auth (JWT), testing | The library ecosystem |

DevOps (Docker, CI/CD, cloud, Kubernetes) is mostly separate from the framework. Ktor shows every piece, which makes it a good teacher; "magic" frameworks hide the pieces.

## Bugs and fixes
| Error or symptom | Cause | Fix |
|---|---|---|
| No request logs after adding the CallLogging dependency | The dependency makes the plugin available; it was not installed | Create `configureMonitoring()` with `install(CallLogging)` and list it in `application.yaml` |
| No `Application.kt` with `main()` | This project has a one-line `main.kt` launcher | `mainClass` is `io.ktor.server.netty.EngineMain`; both paths reach `EngineMain` |
| `Address already in use` | The BE-01 server still held port 8080 | Stop it first |

## Quiz and mistakes
- Score: 3/3 (buttons).
- Adding a library does not enable it.

## In my own words
✏️ Two lines.